In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#import tf
import torch
import time
import math
from IPython.display import clear_output

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv
import tensorflow as tf

# torch.compat.v1.enable_eager_execution()

In [ ]:
### Solver ###
class TGD_class():
    #Initial parameters
    def __init__(self, K, dt,zeta, sigma,D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros):
        self.K = K.type(torch.complex64) 
        self.dt = dt
        self.MAX_ITER = MAX_ITER
        self.unity = unity.to('cuda')
        self.zeros=zeros.to('cuda')
        #scale = torch.dtypes.cast(Get_Scale(K,N), dtype='complex64')
        self.eps_coeff = eps_coeff
        self.rho_thr_coeff = rho_thr_coeff
        self.gamma_init_coeff = gamma_init_coeff #0.2*scale
        self.N=N
        self.attempt_num=attempt_num
        self.pi = np.pi
        self.zeta=zeta
        self.sigma=sigma
        self.D_coeff=D_coeff
        self.scale = torch.max(torch.einsum('ijk->i', torch.abs(self.K)))
#     def Get_Scale(self):
#         isum=torch.dtypes.cast(torch.einsum('ijk->i', torch.abs(self.K)), dtype='float')
#         scale = torch.reduce_max(isum)
#         return scale
    
    def GD_Solver(self, phi_old, phi_new, gamma, eps, rho_thr):
        Density = torch.abs(phi_old)**2
#         print(Density.size(), unity.size(), self.zeros.size(), rho_thr.size())
        D = self.D_coeff*torch.maximum(self.unity-Density/rho_thr,self.zeros) 
        Noise = self.sigma * 1j * math.pi * torch.rand(self.N, self.attempt_num, device = torch.device('cuda'))
        Noise = torch.exp(Noise)               
        dphi = torch.multiply(gamma - Density , phi_old)
        #for y in range(N):
        
        
        
        
        dphi-=self.zeta/self.N*torch.sqrt(rho_thr)*torch.einsum('ijl,js,ls->is', self.K, phi_old, torch.conj(phi_old)) #torch.outer(phi_old,torch.conj(phi_old))
        phi_new = phi_old + self.dt * (dphi + torch.multiply(D,Noise))  
        gamma = gamma + self.dt * eps * (rho_thr * self.unity - Density)
        return phi_new, gamma
    
    
    
    def GD_eval(self):
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        phi_1 = torch.zeros(self.N,self.attempt_num, dtype = torch.complex64, device = torch.device('cuda'))
        phi_2 = phi_1.clone()
        
        gamma = -gamma_init * torch.ones((self.N,self.attempt_num), device = torch.device('cuda'))
        
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K.to(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    
    
    
    
    
    def GD_eval_cc(self, phi_1, gamma, K1,K):
        self.K=K1
        self.K_en=K
        eps =  self.eps_coeff*self.scale
        rho_thr = self.rho_thr_coeff*self.scale
        #gamma_init = self.gamma_init_coeff*self.scale #0.2*scale
        #Initialisation
        #phi_1 = torch.zeros((self.N,self.attempt_num), dtype='complex64')
        phi_2 = phi_1.clone()
        #gamma = -gamma_init * torch.ones((self.N,self.attempt_num), dtype='complex64')
        for m in range(self.MAX_ITER):
            phi_2, gamma = self.GD_Solver(phi_1, phi_2, gamma,  eps, rho_thr)
            phi_1, gamma = self.GD_Solver(phi_2, phi_1, gamma,  eps, rho_thr)
        spins=torch.sign(torch.cos(torch.angle(phi_1)))    
        self.lastcuts= torch.einsum('ijk,is,js,ks->s',self.K_en.type(dtype=torch.float32),spins,spins,spins)
        #
        if torch.sum(torch.isnan(self.lastcuts).type(dtype= torch.float))!=0:
            self.lastcuts=1000*torch.ones(self.attempt_num)
        return self.lastcuts, spins, phi_1, gamma
    

In [ ]:
class TGDInterface(mli.Interface):
        def __init__(self, tgd, attempt_num, clip_percents): 
            super(TGDInterface, self).__init__()
            self.tgd=tgd
            self.attempt_num = attempt_num
            self.tgd.GD_eval()
            #opt_norm=sess.run(tgd.lastcuts)
            self.mean = np.mean(np.array(tgd.lastcuts.cpu()))
            self.scaleen = np.min(np.array(tgd.lastcuts.cpu()))#+self.perc
            self.minimum = np.min(np.array(tgd.lastcuts.cpu()))
            #self.TGD.pump(O,S,D,N, Jmax)

            #self.TGD.pump(O,S,D,N, Jmax)



        def get_sim_params(self, packed_params):
            num_single = 7
            logsig, logdt, logzeta, logeps, logrho, loggamma, logD  = packed_params[:num_single]

            sigma = 10**logsig
            dt = 10**logdt
            zeta = 10**logzeta
            eps_coeff=10**logeps
            rho_thr_coeff=10**logrho
            gamma_init_coeff=10**loggamma
            D_coeff=10**logD
            return sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff


        def get_next_cost_dict(self, params_dict):
            params = params_dict['params']

            sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff = self.get_sim_params(params)

            print('\nTGD:\nsigma={}\ndt={}\nzeta={}\neps_coeff={}\nrho_thr_coeff={}\ngamma_init_coeff={}\nD_coeff={}\n'.format(sigma, dt, zeta, eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff))

            self.tgd.sigma = sigma
            self.tgd.dt = dt
            self.tgd.zeta = zeta
            self.tgd.eps_coeff=eps_coeff
            self.tgd.rho_thr_coeff=rho_thr_coeff
            self.tgd.gamma_init_coeff=gamma_init_coeff
            self.tgd.D_coeff=D_coeff

            #self.TGD.rate=rate
            self.tgd.GD_eval()
            #H_eval=sess.run(self.tgd.lastcuts)
            cost_perc = min([np.mean(np.array(self.tgd.lastcuts.cpu()))/np.abs(self.mean),1]) #np.min(self.TGD.lastcuts)
            cost_min = min([np.min(np.array(self.tgd.lastcuts.cpu()))/np.abs(self.scaleen), 1]) 
            cost=cost_min+0.5*cost_perc
            #cost=min([np.min(self.TGD.lastcuts.cpu())/self.minimum, 1])
            uncer = 0
            bad = False
            print('Initial perc', self.mean)
#             target_perc=self.mean
#             target_scale=self.scale_en
            print('Final perc', np.mean(np.array(self.tgd.lastcuts.cpu())))
            print('Final energy', np.min(np.array(self.tgd.lastcuts.cpu())))
            cost_dict = {'cost':cost, 'uncer':uncer, 'bad':bad}
            return cost_dict

In [ ]:
# attempt_num=2000
MAX_ITER=1000
N = 1
# unity=torch.ones((N,attempt_num),dtype='complex64')
# zeros=torch.zeros((N,attempt_num),dtype='complex64')
# zeta=3/1.2
# sigma=2.0
# D_coeff=100
# eps_coeff=0.01
# rho_thr_coeff=0.1
# gamma_init_coeff=0.2
# dt=0.001

# sigma=1000.0
# dt=0.01861448735320851
# zeta=20.059397383508443
# eps_coeff=0.0001
# rho_thr_coeff=0.009107926652503331
# gamma_init_coeff=0.0006000215691731245
# D_coeff=1.0000061230554544
#a=[]

sigma=48.065791179509
dt=0.0013238314554345335
zeta=0.610248730406827
eps_coeff=1.588285530331451
rho_thr_coeff=3.503384276072272
gamma_init_coeff=0.572423088617843
D_coeff=70.52866057895025

# sigma=527.2921387512716
# dt=0.0021973513463198484
# zeta=0.5208166443684832
# eps_coeff=0.11382807702939306
# rho_thr_coeff=3.5643603768125725
# gamma_init_coeff=0.10265542594579541
# D_coeff=6.272645511285573


# sigma=413.5080438551393
# dt=0.0008903572283027558
# zeta=0.509867575528195
# eps_coeff=0.003226045278516126
# rho_thr_coeff=3.26118137607805
# gamma_init_coeff=0.06194152406047089
# D_coeff=1.9524381020135582

sigma=2.835269378718527
dt=0.0006829170080534533
zeta=22.69041999498189
eps_coeff=0.0018539600931654447
rho_thr_coeff=0.057567955466196676
gamma_init_coeff=0.09123284962233773
D_coeff=35.47046565639147

sigma=19.261170984818033
dt=8.002945022148085e-05
zeta=1.743536426947582
eps_coeff=0.019560868652868123
rho_thr_coeff=5.118683230467833
gamma_init_coeff=0.17483829886030414
D_coeff=34.47324602399852

sigma=2.0
dt=0.001
zeta=3.0/1.2
eps_coeff=0.01
rho_thr_coeff=0.1
gamma_init_coeff=0.2
D_coeff=100.0

sigma=1.5593756737678446
dt=0.0008011837387141064
zeta=14.280441742760813
eps_coeff=0.006444435545826028
rho_thr_coeff=6.730384070279093
gamma_init_coeff=0.23585717760787595
D_coeff=528.9853625231918
##sparse_10
sigma=11.21704112508936
dt=0.01886374625999647
zeta=1.9798467323712294
eps_coeff=0.012727639052259484
rho_thr_coeff=2.980642780944409
gamma_init_coeff=0.037531137829078964
D_coeff=65.69330545587947

## sparse_20
sigma=37.194237141725694
dt=0.047159283942434835
zeta=1.7531997986283467
eps_coeff=0.0021377207550303833
rho_thr_coeff=0.19162988880031392
gamma_init_coeff=0.003538091417301595
D_coeff=15.502019054995436

# ##sparse_40
# sigma=1.2617439969550261
# dt=0.04547831402654722
# zeta=1.1545027657751108
# eps_coeff=0.08162775618552788
# rho_thr_coeff=0.08499571118245709
# gamma_init_coeff=0.0034371496648554906
# D_coeff=68.92712091148903

##Sparse_40_4
sigma=100.0
dt=0.027196751756440356
zeta=3.5808576313907365
eps_coeff=0.007843064307644618
rho_thr_coeff=0.0623304540949191
gamma_init_coeff=0.0026009272411341698
D_coeff=19.983084483324067

## Sparse_60
sigma=1.0
dt=0.0012151262501697874
zeta=0.7786768405186959
eps_coeff=0.45704409287902514
rho_thr_coeff=1.610478710511829
gamma_init_coeff=0.001
D_coeff=100.0
# ## Sparse 100
# sigma=5.883428728352809
# dt=0.0991404657291636
# zeta=0.6639746262219267
# eps_coeff=0.00355995140869215
# rho_thr_coeff=0.013230352355553528
# gamma_init_coeff=0.0019119239072552434
# D_coeff=2.1873234793673966

# ##SK10_4
# sigma=3.314537861809106
# dt=0.002260695206391212
# zeta=0.44838095962357793
# eps_coeff=0.024916945276448842
# rho_thr_coeff=0.12008790436060866
# gamma_init_coeff=0.025294606483793537
# D_coeff=10.178965773177131

# ##SK20_4
# sigma=2.1777181350782828
# dt=0.005276559531533092
# zeta=0.15303951421963394
# eps_coeff=0.001
# rho_thr_coeff=0.15732791156769316
# gamma_init_coeff=0.46525106380666315
# D_coeff=42.93859785836815


# ##SK30_4
# sigma=1.0
# dt=0.014268816154700456
# zeta=1.2436772851683142
# eps_coeff=0.001
# rho_thr_coeff=0.001
# gamma_init_coeff=0.0032797460557905684
# D_coeff=9.702760191815358

# sigma=1.1777181350782828
# dt=0.005276559531533092
# zeta=0.15303951421963394
# eps_coeff=0.001
# rho_thr_coeff=0.15732791156769316
# gamma_init_coeff=0.46525106380666315
# D_coeff=42.93859785836815


# ##SK40_4
# sigma=1.1079150974087375
# dt=0.0010008253612745197
# zeta=0.03938163709205323
# eps_coeff=0.1999354010519413
# rho_thr_coeff=0.03901895140583021
# gamma_init_coeff=0.11081345773363177
# D_coeff=1.0
# sigma=1.0
# dt=0.0026630403199687934
# zeta=0.19335000576325634
# eps_coeff=0.14721646201498448
# rho_thr_coeff=0.007113412159007323
# gamma_init_coeff=0.001
# D_coeff=1.0


# ###Dense 10
# sigma=1.693915695855958
# dt=0.006906081809040627
# zeta=0.697356595556547
# eps_coeff=0.001
# rho_thr_coeff=0.5370313712981636
# gamma_init_coeff=0.00983455205099063
# D_coeff=100.0

# ## Dense 10
# sigma=10.924102559621003
# dt=0.006787521129000991
# zeta=0.5641639902889855
# eps_coeff=0.001
# rho_thr_coeff=0.8613142862625329
# gamma_init_coeff=0.003487071135460135
# D_coeff=100.0

# ## Dense 20

# sigma=100.0
# dt=0.001
# zeta=0.6461934926572646
# eps_coeff=0.0030533590635718985
# rho_thr_coeff=0.7534715732111237
# gamma_init_coeff=0.011848318283595002
# D_coeff=42.92462042659285


# #Dense 40
# sigma=100.0
# dt=0.001
# zeta=0.3196516005978744
# eps_coeff=0.0012582828886735099
# rho_thr_coeff=0.6548614171603169
# gamma_init_coeff=0.00997416761051083
# D_coeff=35.301873145812706


# ##Dense 60
# sigma=54.30553598634419
# dt=0.0046308095740240154
# zeta=1.179919981745229
# eps_coeff=0.0012262317190383414
# rho_thr_coeff=0.007900506702910266
# gamma_init_coeff=0.005461665021975287
# D_coeff=87.30477917426751


# ## Dense 100
# sigma=96.77833419445527
# dt=0.001
# zeta=0.4494495885562207
# eps_coeff=0.001
# rho_thr_coeff=0.05206946856129836
# gamma_init_coeff=0.001
# D_coeff=10.0

sigma=55.877559779376675
dt=0.014409900159932964
zeta=0.3169948402015307
eps_coeff=0.001
rho_thr_coeff=0.001
gamma_init_coeff=0.006526521135187492
D_coeff=27.516960058326774
for p, number in zip([5,7,8,3,7,7], [0,4,5,6,7,9]):

    psize=int(10*(p+1))
    N = psize
#     unity=torch.ones((N,attempt_num),dtype='complex64')
#     zeros=torch.zeros((N,attempt_num),dtype='complex64')
#     zeta=3/1.2
#     sigma=2.0
#     D_coeff=100
#     eps_coeff=0.01
#     rho_thr_coeff=0.1
#     gamma_init_coeff=0.2
#     dt=0.001
#     sigma=1000.0
#     dt=7.848602059628158e-06
#     zeta=2.00700961965306
#     eps_coeff=0.024449116728533365
#     rho_thr_coeff=6.328678877168711
#     gamma_init_coeff=0.013668420747450831
#     D_coeff=35.92497962406887
#     sigma=1.98491155384835
#     dt=9.818421290427106e-05
#     zeta=5.681876723804659
#     eps_coeff=22.124922601778955
#     rho_thr_coeff=0.02760111823011615
#     gamma_init_coeff=1.651f5030846721428e-06
#     D_coeff=521.126826197589

#     sigma=1.1256007415673353
#     dt=0.0006179019314898143
#     zeta=4.834222212129382
#     eps_coeff=71.16149654244549
#     rho_thr_coeff=0.046610184506237516
#     gamma_init_coeff=1.0221895856253064e-06
#     D_coeff=17.19357320984846

#     sigma=0.3002947522312934
#     dt=0.0038778531707231575
#     zeta=0.427088618700813
#     eps_coeff=0.019260895614638177
#     rho_thr_coeff=0.5068383055389816
#     gamma_init_coeff=0.0002918978265413923
#     D_coeff=2.5547710225981537


#     sigma=4.390036180335123
#     dt=0.0004027187443314652
#     zeta=1.7889439353116543
#     eps_coeff=0.00010566437989167337
#     rho_thr_coeff=0.007927963823685368
#     gamma_init_coeff=0.0001
#     D_coeff=9.663415239891606

    sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff = np.load('GD_each/'+str(psize)+'_'+str(number)+'_params_tuned.npy')

#         for jj in range(2):


    foldername = str(psize)+'_'+str(number)        #K_npy=np.load(foldername+'.npy').astype'complex64'
    #K_npy=np.load(foldername+'.npy').astype'complex64'
    K = torch.from_numpy(np.load(foldername+'.npy')).to('cuda').type(dtype = torch.float)
    K1 = np.load(foldername+'.npy').astype('complex64')

    for i in range(0,np.shape(K1[0,0,:])[0]):
        for j in range(i+1,np.shape(K1[0,0,:])[0]):
            for k in range(j+1,np.shape(K1[0,0,:])[0]):
                  if (np.random.rand(1)[0]>(1-np.shape(K1[0,0,:])[0]/50/100)):
                    K1[i,j,k] = K[i,j,k]*(1.0+3.0*1j)
                    K1[i,k,j] = K1[i,j,k]
                    K1[j,i,k] = K1[i,j,k]
                    K1[j,k,i] = K1[i,j,k]
                    K1[k,i,j] = K1[i,j,k]
                    K1[k,j,i] = K1[i,j,k]
    K1_ten=torch.from_numpy(K1).to('cuda')
#             sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff=np.load(foldername+'params_tuned_Berlova.npy')
#             sigma/=10
#             if number==0:
    attempt_num=500
    unity=torch.ones((N,attempt_num))
    zeros=torch.zeros((N,attempt_num))
    clip_percents=50
    population_size=7
    percentile = 50
#             foldername = str(psize)+'_'+str(9)
    #K_npy=np.load(foldername+'.npy').astype'complex64'
#             K= torch.from_numpy(np.load(foldername+'.npy')).to('cuda')
    TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff,MAX_ITER, N, attempt_num, unity, zeros)
    interface = TGDInterface(TGD,attempt_num,percentile)

    first_params = np.array([np.log10(TGD.sigma), np.log10(TGD.dt), np.log10(TGD.zeta), np.log10(TGD.eps_coeff), np.log10(TGD.rho_thr_coeff), np.log10(TGD.gamma_init_coeff), np.log10(TGD.D_coeff)])
    min_boundary = np.array([0,-6,-4,-4,-4, -4,0])
    max_boundary = np.array([ 3, 0, 3, 3, 3, 3, 2])
    num_params = 7
    config = dict(controller_type='neural_net', 
                  training_type = 'differential_evolution',
                  first_params = first_params,
                  min_boundary = min_boundary, 
                  max_boundary = max_boundary,
                  num_params = num_params,
                  trust_region = 0.5,#0.5
                  max_num_runs = 1500,
                  num_training_runs = 2 * num_params,
                  log_level = 0, 
                  no_delay = False, 
                  max_num_runs_without_better_params = 150, 
                  #evolution_strategy='best1', 
                  population_size=10, 
                  mutation_scale = (0.5,1.0), 
                  cross_over_probability = 0.7, 
                  default_bad_cost = 1.05, 
                  #target_cost=-(8+np.percentile(np.array(TGD.lastcuts.cpu()), clip_percents))/(-np.min(np.array(TGD.lastcuts.cpu()))+np.percentile(np.array(TGD.lastcuts.cpu()), clip_percents)),
                  default_bad_uncertainty = 0.01,
                  cost_has_noise = True)

    controller = mlc.create_controller(interface, **config)
    controller.optimize()

    #clear_output(True)

    print('Best cut found:')
    #print(-controller.best_cost*controller.interface.scale+controller.interface.perc)

    mlv.show_all_default_visualizations(controller)
    sigma=10**controller.best_params[0]
    dt=10**controller.best_params[1]
    zeta=10**controller.best_params[2]
    eps_coeff=10**controller.best_params[3]
    rho_thr_coeff=10**controller.best_params[4]
    gamma_init_coeff=10**controller.best_params[5]
    D_coeff=10**controller.best_params[6]   

    foldername = str(psize)+'_'+str(number)       #K_npy=np.load(foldername+'.npy').astype'complex64'
    #K_npy=np.load(foldername+'.npy').astype'complex64'
    K = torch.from_numpy(np.load(foldername+'.npy')).to('cuda').type(dtype = torch.float)
    attempt_num=2000
    params=[sigma, dt, zeta,  eps_coeff, rho_thr_coeff, gamma_init_coeff, D_coeff]
    print(p, number)
    unity=torch.ones((N,attempt_num),  device = torch.device('cuda'))
    zeros=torch.zeros((N,attempt_num), device = torch.device('cuda'))
    TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros)
    H_opt1, spins1, phi_1, gamma=TGD.GD_eval()
    H_opt1_nump=H_opt1.cpu().numpy()
    plt.hist(H_opt1_nump, color='blue', alpha=0.5)
    plt.show()
    H_opt2, spins2, casda, asdasd=TGD.GD_eval_cc(phi_1, gamma, K1_ten,K)
    H_opt2_nump=H_opt2.cpu().numpy()
    while  np.sum(H_opt2_nump==2000)>0 or np.sum(H_opt1_nump==2000)>0:
        H_opt1, spins1, phi_1, gamma=TGD.GD_eval()
        H_opt1_nump=H_opt1.cpu().numpy()
        print(phi_1.cpu().numpy().max(), gamma.cpu().numpy().max())
        #sigma*=1.1
        #dt/=1.1
        #phi_1/=1.1
        #gamma/=1.1
        TGD=TGD_class(K,dt, zeta, sigma, D_coeff, gamma_init_coeff, rho_thr_coeff,eps_coeff, MAX_ITER, N, attempt_num, unity, zeros)
        H_opt2, spins2, casda, asdas=TGD.GD_eval_cc(phi_1, gamma, K1_ten,K)
        H_opt2_nump=H_opt2.cpu().numpy()
    plt.hist(H_opt1_nump, color='blue', alpha=0.5)
    plt.hist(H_opt2_nump, color='red', alpha=0.5)
    plt.show()
    print(H_opt1_nump.mean(), H_opt2_nump.mean())
    print(H_opt1_nump.min(), H_opt2_nump.min())
    H_opt=H_opt1_nump*(H_opt1_nump<=H_opt2_nump)+H_opt2_nump*(H_opt1_nump>H_opt2_nump)
#             a=np.append(a,H_opt)
    spins = np.array(spins1.cpu()) * (H_opt1_nump<=H_opt2_nump).reshape(1,-1) + np.array(spins2.cpu()) *(H_opt1_nump>H_opt2_nump).reshape(1,-1)

    np.save('GD_each/'+str(psize)+'_'+str(number)+'_bit.npy', spins)
    np.save('GD_each/'+str(psize)+'_'+str(number)+'.npy', H_opt)
    np.save('GD_each/'+str(psize)+'_'+str(number)+'_params_tuned.npy', params)
np.save(foldername+'energy_tuned_Berlova_cc2.npy', a)
np.save(foldername+'params_tuned_Berlova.npy', params)
        
            
        
#### Parameters
### Sparse
##6
# sigma=1000.0
# dt=0.0021426195743340713
# zeta=1.0906444813395457
# eps_coeff=1e-06
# rho_thr_coeff=2.1101693638825725
# gamma_init_coeff=2.4979872199216417e-06
# D_coeff=1000.0
##7
# sigma=1000.0
# dt=0.00018460539824748576
# zeta=9.429455343792
# eps_coeff=3.207542559776922e-06
# rho_thr_coeff=0.20195032362854753
# gamma_init_coeff=2.4979700144983255e-06
# D_coeff=333.4234962000213
# ##8
# sigma=3.859248443765062
# dt=5.153028794337006e-05
# zeta=5.045079687380809
# eps_coeff=0.05267078423387429
# rho_thr_coeff=0.90344571910411
# gamma_init_coeff=1.797381738169602e-05
# D_coeff=5.948851273968465
# ##9
# sigma=1.4508587561467412
# dt=0.0010408196623105973
# zeta=3.530004486402835
# eps_coeff=0.011664717445570887
# rho_thr_coeff=0.17483349443306614
# gamma_init_coeff=0.12430897723074297
# D_coeff=1.004835359599058
# ##10
# sigma=1.3359832333900359
# dt=0.0010408194710289766
# zeta=3.530004486402806
# eps_coeff=0.011664717445570887
# rho_thr_coeff=0.2623002851744743
# gamma_init_coeff=0.12430897723074297
# D_coeff=1.0